### Cell 1

Load the Digikala comments dataset from the source CSV file.

In [9]:
import pandas as pd
from pathlib import Path
import re

comments_path = r"E:\Work\AI\MAKTAB\HW-CW-S02\NLP Project\digikala-comments.csv"


df_comments = pd.read_csv(comments_path, low_memory=True)

### Cell 2

Select the relevant columns, normalize Persian text, combine titles and bodies, and remove empty records.

In [10]:
df = df_comments[
    ["id","title", "body", "product_id","rate", "recommendation_status"]
].copy()

                    
for col in ["title", "body"]:
    df[col] = df[col].fillna("").astype(str)

def normalize_text(text):
    text = text.replace("ي", "ی")
    text = text.replace("ك", "ک")
    text = text.replace("\u200c", " ")

                         
    text = re.sub(r"\.{2,}", " ", text)

                         
    text = re.sub(r"\s+", " ", text)

    return text.strip()

                         
df["title_norm"] = df["title"].apply(normalize_text)
df["body_norm"] = df["body"].apply(normalize_text)

                    
df["text"] = (
    df["title_norm"] + " " + df["body_norm"]
).str.strip()

                                                    
df["text"] = df["text"].str.strip(" .،؛")

                                   
before = len(df)

df = df[df["text"] != ""].copy()

print("Initial record count:", before)
print("Records removed due to empty text:", before - len(df))
print("Remaining records:", len(df))

display(df[["title", "body", "text"]].head(10))

Initial record count: 6156289
Records removed due to empty text: 228
Remaining records: 6156061


,title,body,text
0,پیشنهاد نمیشود,به درد نمیخوره,پیشنهاد نمیشود به درد نمیخوره
1,بسته بندی بد,می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بن...,بسته بندی بد می تونست به عنوان یه کالای فرهنگی...
2,برس ریمل,بسته بندیش خوب بود\r\n کاربرد و کیفیتشم خیلی خ...,برس ریمل بسته بندیش خوب بود کاربرد و کیفیتشم خ...
3,خوبه و خوشرنگ,به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد ...,خوبه و خوشرنگ به نظرم خوبه فقط یکم ظریفه. از ر...
4,برس رنگ مو,معمولیه اگه واسه خونه رنگ کردن شخصی میخواین او...,برس رنگ مو معمولیه اگه واسه خونه رنگ کردن شخصی...
5,خوبه,قبلا هم استفاده کردم اگه بلد باشین کار کردن با...,خوبه قبلا هم استفاده کردم اگه بلد باشین کار کر...
6,بدک نیست,خوبه,بدک نیست خوبه
7,پد شستشوی براش,خیلی به کارتون میاد قیمتشم مناسبه,پد شستشوی براش خیلی به کارتون میاد قیمتشم مناسبه
8,لنز ناخون,دقیقا مطابق عکس بود,لنز ناخون دقیقا مطابق عکس بود
9,خوب,برا حالت دادن با اسپری یا پودر عالیه,خوب برا حالت دادن با اسپری یا پودر عالیه


### Cell 3

Refine the text construction to avoid duplicating identical titles and bodies, then remove empty records.

In [11]:
title_equals_body = (
    df["title_norm"].ne("") &
    df["title_norm"].eq(df["body_norm"])
)


df["text"] = (
    df["body_norm"].where(
        title_equals_body,
        df["title_norm"] + " " + df["body_norm"]
    )
).str.strip()


df["text"] = df["text"].str.strip(" .،؛")


before = len(df)
df = df[df["text"] != ""].copy()

print("Initial record count:", before)
print("Records removed due to empty text:", before - len(df))
print("Remaining records:", len(df))
print("Rows where title and body were identical:",
      int(title_equals_body.loc[df.index].sum()))

display(df[["title", "body", "text"]].head(10))

Initial record count: 6156061
Records removed due to empty text: 0
Remaining records: 6156061
Rows where title and body were identical: 133549


,title,body,text
0,پیشنهاد نمیشود,به درد نمیخوره,پیشنهاد نمیشود به درد نمیخوره
1,بسته بندی بد,می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بن...,بسته بندی بد می تونست به عنوان یه کالای فرهنگی...
2,برس ریمل,بسته بندیش خوب بود\r\n کاربرد و کیفیتشم خیلی خ...,برس ریمل بسته بندیش خوب بود کاربرد و کیفیتشم خ...
3,خوبه و خوشرنگ,به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد ...,خوبه و خوشرنگ به نظرم خوبه فقط یکم ظریفه. از ر...
4,برس رنگ مو,معمولیه اگه واسه خونه رنگ کردن شخصی میخواین او...,برس رنگ مو معمولیه اگه واسه خونه رنگ کردن شخصی...
5,خوبه,قبلا هم استفاده کردم اگه بلد باشین کار کردن با...,خوبه قبلا هم استفاده کردم اگه بلد باشین کار کر...
6,بدک نیست,خوبه,بدک نیست خوبه
7,پد شستشوی براش,خیلی به کارتون میاد قیمتشم مناسبه,پد شستشوی براش خیلی به کارتون میاد قیمتشم مناسبه
8,لنز ناخون,دقیقا مطابق عکس بود,لنز ناخون دقیقا مطابق عکس بود
9,خوب,برا حالت دادن با اسپری یا پودر عالیه,خوب برا حالت دادن با اسپری یا پودر عالیه


### Cell 4

Create a trusted subset of texts based on majority-label agreement and save the cleaned dataset.

In [12]:
labels = df["recommendation_status"].astype("string").str.strip()
valid = df.loc[labels.notna() & labels.ne("") , ["text"]].copy()
valid["label"] = labels[labels.notna() & labels.ne("")]

label_dist = (
    valid.groupby(["text", "label"])
         .size()
         .unstack(fill_value=0)
)

                        
majority_label = label_dist.idxmax(axis=1)
agreement = label_dist.max(axis=1) / label_dist.sum(axis=1)

                                           
trusted_texts = agreement[agreement >= 0.80].index
label_map = majority_label.loc[trusted_texts]

df_clean = df[df["text"].isin(trusted_texts)].copy()
df_clean["recommendation_status_clean"] = df_clean["text"].map(label_map)


before_duplicates = len(df_clean)

df_clean = df_clean.drop_duplicates().copy()

print("Number of completely duplicate rows removed:",before_duplicates - len(df_clean))   

print("Initial records:", len(df))
print("Remaining records:", len(df_clean))
print("Removed records:", len(df) - len(df_clean))
print("Unique trusted texts:", len(trusted_texts))

df_clean.to_csv("digikala_comments_clean_ids.csv", index=False)

Number of completely duplicate rows removed: 2558
Initial records: 6156061
Remaining records: 5371359
Removed records: 784702
Unique trusted texts: 3903633
